# 知識蒸餾 Knowledge Distillation

訓練時可以用很大的模型，或是很多個模型的組合，因為訓練不必即時完成；
部署給大量使用者時，模型必須小而快。**知識蒸餾**是把大模型學到的東西移轉給小模型的方法：
小模型除了學正確答案，也學大模型對每一個類別給出的機率。

本筆記本依據 Hinton、Vinyals、Dean 在 2015 年發表的論文〈Distilling the Knowledge in a Neural Network〉，分八段：

| 段 | 內容 |
|---|---|
| 1 | 載入資料 |
| 2 | 溫度軟化的 softmax：手算例 |
| 3 | 蒸餾的損失函數與梯度：手算例，並用自動微分核對 |
| 4 | 訓練老師模型 |
| 5 | 學生模型：只學正確答案，與蒸餾比較 |
| 6 | 更小的學生：溫度要取中間值 |
| 7 | 學生從沒看過數字 3 |
| 8 | 結果整理 |

**執行方式**：由上往下逐格執行。想快一點可以先換 GPU（「執行階段 → 變更執行階段類型 → T4 GPU」）；
不換也能跑，CPU 全部跑完約需數分鐘。

## 1. 載入資料

資料是手寫數字 MNIST 的一部分：訓練用 10,000 張（0 到 9 各 1,000 張），測試用 10,000 張（完整的測試集）。
每張 28×28 灰階，像素值 0 到 255。檔名 `kd_mnist.npz`，約 3.3 MB。

取得資料有兩種方式，下面這一格會自動判斷：

1. **拖進 Colab**：把老師提供的 `kd_mnist.npz` 拖到 Colab 左側的「檔案」區，放在預設資料夾 `/content`。
2. **從課程 repo 讀取**：左側沒有這個檔時，程式從課程的 GitHub repo 下載同一個檔。

In [ ]:
import os
import urllib.request

import numpy as np

np.set_printoptions(suppress=True)  # 小數不用科學記號，和簡報上的寫法一致

DATA = 'kd_mnist.npz'
URL = ('https://raw.githubusercontent.com/pychang-ai/115-1_DLMA/main/'
       'topics/knowledge-distillation/kd_mnist.npz')

if os.path.exists(DATA):
    print('使用已上傳的', DATA)
else:
    print('左側沒有', DATA, '，從課程 repo 下載')
    urllib.request.urlretrieve(URL, DATA)

d = np.load(DATA)
x_train = d['x_train'].reshape(-1, 784).astype('float32') / 255.0
y_train = d['y_train'].astype('int32')
x_test = d['x_test'].reshape(-1, 784).astype('float32') / 255.0
y_test = d['y_test'].astype('int32')
print('訓練', x_train.shape, '測試', x_test.shape)
print('訓練集各數字張數', np.bincount(y_train))
print('測試集各數字張數', np.bincount(y_test))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for k, ax in enumerate(axes):
    i = np.where(y_train == k)[0][0]
    ax.imshow(x_train[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(str(k))
    ax.axis('off')
plt.show()

## 2. 溫度軟化的 softmax

神經網路最後一層輸出的數字稱為 logit，記為 $z_i$。softmax 把 logit 轉成機率，論文的式 (1) 多了一個溫度 $T$：

$$q_i = \frac{\exp(z_i/T)}{\sum_j \exp(z_j/T)}$$

$T = 1$ 就是一般的 softmax。$T$ 越大，各類別的機率越接近，分布越「軟」。

**手算例**：一張手寫的 2，老師模型對「2、3、7」三個類別的 logit 為 $v = (8, 4, 0)$。
分四步算：除以 $T$、取指數、加總、各自除以總和。

In [ ]:
def softmax_T(z, T):
    """論文式 (1)：溫度 T 的 softmax，逐步印出每一步。"""
    z = np.asarray(z, dtype='float64')
    a = z / T
    e = np.exp(a)
    s = e.sum()
    return a, e, s, e / s


v = [8.0, 4.0, 0.0]
for T in [1, 4]:
    a, e, s, p = softmax_T(v, T)
    print(f'T = {T}')
    print('  第 1 步 z/T      ', np.round(a, 4))
    print('  第 2 步 exp(z/T) ', np.round(e, 4))
    print('  第 3 步 加總      ', round(s, 4))
    print('  第 4 步 機率      ', np.round(p, 4))
    print(f'  「3」與「7」的機率比 {p[1] / p[2]:.2f}')

$T = 1$ 時，「3」和「7」的機率是 0.0180 與 0.0003，都接近 0；訓練學生時，這兩個數字對交叉熵幾乎沒有影響。
$T = 4$ 時變成 0.2447 與 0.0900，「這張 2 比較像 3、不太像 7」這個資訊才會影響訓練。
這種由老師在高溫下給出的機率，稱為**軟目標** soft targets；正確答案本身（2 的位置是 1，其餘是 0）稱為**硬目標** hard targets。

In [ ]:
labels = ['2', '3', '7']
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), sharey=True)
for ax, T in zip(axes, [1, 4]):
    p = softmax_T(v, T)[3]
    ax.bar(labels, p, color=['#1F4E79', '#C03030', '#59636E'])
    for k, val in enumerate(p):
        ax.text(k, val + 0.02, f'{val:.4f}', ha='center')
    ax.set_title(f'teacher softmax, T = {T}')
    ax.set_ylim(0, 1.1)
plt.show()

## 3. 蒸餾的損失函數與梯度

學生的損失由兩項加權組成：

- **軟目標項**：老師在溫度 $T$ 下的機率 $p$，與學生在**同一個溫度**下的機率 $q$ 的交叉熵 $C_{soft} = -\sum_i p_i \log q_i$。
- **硬目標項**：正確答案與學生在 $T = 1$ 下的機率的交叉熵 $C_{hard}$。

$$L = \alpha \, T^2 \, C_{soft} + (1 - \alpha) \, C_{hard}$$

論文指出軟目標項的梯度大小與 $1/T^2$ 成正比，所以乘上 $T^2$，換溫度時兩項的比重才不會跟著變。
論文也指出硬目標的權重取小一些效果較好，本筆記本取 $\alpha = 0.9$。

軟目標項對學生 logit $z_i$ 的梯度是論文的式 (2)：

$$\frac{\partial C_{soft}}{\partial z_i} = \frac{1}{T}(q_i - p_i)$$

**手算例**：同一張 2，學生的 logit 為 $z = (4, 2, 2)$，溫度 $T = 4$，$\alpha = 0.9$。

In [ ]:
T, alpha = 4.0, 0.9
z = np.array([4.0, 2.0, 2.0])
p = softmax_T(v, T)[3]          # 老師的軟目標
q = softmax_T(z, T)[3]          # 學生在 T = 4
q1 = softmax_T(z, 1)[3]         # 學生在 T = 1
y = np.array([1.0, 0.0, 0.0])   # 正確答案是 2

c_soft = -(p * np.log(q)).sum()
c_hard = -(y * np.log(q1)).sum()
loss = alpha * T ** 2 * c_soft + (1 - alpha) * c_hard
print('老師軟目標 p      ', np.round(p, 4))
print('學生 q（T=4）     ', np.round(q, 4))
print('學生 q（T=1）     ', np.round(q1, 4))
print(f'C_soft = {c_soft:.4f}   C_hard = {c_hard:.4f}')
print(f'L = {alpha} × {T:g}² × {c_soft:.4f} + {1 - alpha:.1f} × {c_hard:.4f} = {loss:.4f}')

g_soft = (q - p) / T            # 式 (2)
g_hard = q1 - y                 # 一般交叉熵的梯度
g = alpha * T ** 2 * g_soft + (1 - alpha) * g_hard
print()
print('式 (2) 軟目標梯度 (q - p)/T ', np.round(g_soft, 4))
print('硬目標梯度 q1 - y           ', np.round(g_hard, 4))
print('總梯度                      ', np.round(g, 4))

用 TensorFlow 的自動微分再算一次，數字要和上面的公式一致：

In [ ]:
import tensorflow as tf

zt = tf.Variable(z)
with tf.GradientTape() as tape:
    qt = tf.nn.log_softmax(zt / T)
    q1t = tf.nn.log_softmax(zt)
    L = alpha * T ** 2 * -tf.reduce_sum(p * qt) + (1 - alpha) * -tf.reduce_sum(y * q1t)
print('自動微分的損失', round(float(L), 4))
print('自動微分的梯度', np.round(tape.gradient(L, zt).numpy(), 4))

**為什麼乘 $T^2$**：同一組 logit，溫度越高，式 (2) 的梯度越小。下表比較 $T = 1, 4, 20$：

In [ ]:
for t in [1, 4, 20]:
    gs = (softmax_T(z, t)[3] - softmax_T(v, t)[3]) / t
    print(f'T = {t:>2}  梯度最大絕對值 {np.abs(gs).max():.6f}   乘 T² 後 {np.abs(gs).max() * t * t:.4f}')

$T$ 從 4 升到 20，梯度縮小約 20 倍，接近 $(20/4)^2 = 25$ 倍；乘上 $T^2$ 之後是 0.85 與 1.05，大小相近。
這個 $1/T^2$ 的比例是高溫下的近似，$T = 1$ 不在這個範圍內。

**高溫下等於比對 logit**：把老師與學生的 logit 各自減去平均值（論文稱 zero-meaned），
溫度遠大於 logit 時，式 (2) 近似論文的式 (4)：

$$\frac{\partial C_{soft}}{\partial z_i} \approx \frac{1}{N T^2}(z_i - v_i)$$

$N$ 是類別數。這等於最小化 $\frac{1}{2}(z_i - v_i)^2$，也就是讓學生的 logit 直接去貼老師的 logit。

In [ ]:
N, T = 3, 20.0
zc, vc = z - z.mean(), np.array(v) - np.mean(v)
exact = (softmax_T(zc, T)[3] - softmax_T(vc, T)[3]) / T
approx = (zc - vc) / (N * T ** 2)
print('減去平均後  z =', np.round(zc, 4), '  v =', np.round(vc, 4))
print('式 (2) 精確值', np.round(exact, 5))
print('式 (4) 近似值', np.round(approx, 5))

## 4. 訓練老師模型

老師是兩層各 1,200 個神經元的全連接網路，用兩種方法防止過擬合，和論文相同：
Dropout，以及把輸入影像隨機平移最多 2 個像素。輸出層不接 softmax，直接輸出 logit，溫度在後面另外處理。

In [ ]:
import keras
from keras import layers

keras.utils.set_random_seed(1)


def mlp(units, regularize=False, name='mlp'):
    """兩個隱藏層的全連接網路，輸出 10 個 logit。regularize=True 時加平移與 Dropout。"""
    inp = keras.Input((784,))
    x = inp
    if regularize:
        x = layers.Reshape((28, 28, 1))(x)
        x = layers.RandomTranslation(2 / 28, 2 / 28, fill_mode='constant')(x)
        x = layers.Flatten()(x)
        x = layers.Dropout(0.2)(x)
    for _ in range(2):
        x = layers.Dense(units, activation='relu')(x)
        if regularize:
            x = layers.Dropout(0.5)(x)
    out = layers.Dense(10)(x)
    return keras.Model(inp, out, name=name)


def test_errors(model, bias=None):
    """回傳測試集答錯張數與 logit。bias 是加在各類別 logit 上的偏移量。"""
    logits = model.predict(x_test, batch_size=2000, verbose=0)
    if bias is not None:
        logits = logits + bias
    return int((logits.argmax(1) != y_test).sum()), logits


teacher = mlp(1200, regularize=True, name='teacher')
teacher.compile(keras.optimizers.Adam(1e-3),
                keras.losses.SparseCategoricalCrossentropy(from_logits=True))
teacher.fit(x_train, y_train, batch_size=128, epochs=30, verbose=0)
teacher_err, _ = test_errors(teacher)
print(f'老師：參數 {teacher.count_params():,} 個，測試集 10,000 張答錯 {teacher_err} 張')

# 老師在移轉集（就是訓練集）上的 logit，之後每個學生都用這一份
v_train = teacher.predict(x_train, batch_size=2000, verbose=0)

## 5. 學生模型：只學正確答案，與蒸餾比較

學生是兩層各 100 個神經元、沒有任何正則化的小網路。同一個學生架構訓練多次：

- **只學正確答案**：$\alpha = 0$，只有硬目標項。
- **蒸餾**：$\alpha = 0.9$，溫度取 1、2、4、8、20。

訓練迴圈依第 3 段的損失函數自己寫，每一次都用相同的初始權重與資料順序，差別只在損失函數。

In [ ]:
def train_student(units, T, alpha, mask=None, epochs=30, seed=2):
    """依 L = α T² C_soft + (1 − α) C_hard 訓練學生。mask 用來從移轉集拿掉部分資料。"""
    keras.utils.set_random_seed(seed)
    student = mlp(units, name='student')
    opt = keras.optimizers.Adam(1e-3)
    x, yy, vv = (x_train, y_train, v_train) if mask is None else \
        (x_train[mask], y_train[mask], v_train[mask])
    ds = tf.data.Dataset.from_tensor_slices((x, yy, vv)).shuffle(len(x), seed=seed).batch(128)

    @tf.function
    def step(xb, yb, vb):
        with tf.GradientTape() as tape:
            zb = student(xb, training=True)
            pb = tf.nn.softmax(vb / T)                                     # 老師的軟目標
            c_soft = -tf.reduce_mean(tf.reduce_sum(pb * tf.nn.log_softmax(zb / T), 1))
            c_hard = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
                labels=yb, logits=zb))
            loss = alpha * T * T * c_soft + (1 - alpha) * c_hard
        grads = tape.gradient(loss, student.trainable_variables)
        opt.apply_gradients(zip(grads, student.trainable_variables))

    for _ in range(epochs):
        for xb, yb, vb in ds:
            step(xb, yb, vb)
    return student


TEMPS = [1, 2, 4, 8, 20]
results = {}
s = train_student(100, 1.0, 0.0)
results[(100, 'hard')] = test_errors(s)[0]
print(f'學生 100：只學正確答案，答錯 {results[(100, "hard")]} 張')
for t in TEMPS:
    s = train_student(100, float(t), 0.9)
    results[(100, t)] = test_errors(s)[0]
    print(f'學生 100：蒸餾 T = {t:>2}，答錯 {results[(100, t)]} 張')

## 6. 更小的學生：溫度要取中間值

論文指出：學生每層有 300 個以上神經元時，8 以上的溫度結果都差不多；學生縮小到每層 30 個時，
2.5 到 4 的溫度明顯比更高或更低的溫度好。原因是溫度越高，越接近第 3 段的「比對 logit」，
連老師那些非常負的 logit 也要學生跟著學，太小的學生學不來。下面用每層 30 個神經元的學生重做一次：

In [ ]:
s = train_student(30, 1.0, 0.0)
results[(30, 'hard')] = test_errors(s)[0]
print(f'學生 30：只學正確答案，答錯 {results[(30, "hard")]} 張')
for t in TEMPS:
    s = train_student(30, float(t), 0.9)
    results[(30, t)] = test_errors(s)[0]
    print(f'學生 30：蒸餾 T = {t:>2}，答錯 {results[(30, t)]} 張')

## 7. 學生從沒看過數字 3

論文把移轉集裡所有的 3 拿掉，學生訓練時一張 3 都沒看過。
只學正確答案的學生不可能答對 3；蒸餾的學生卻能從老師的軟目標學到「這張 8 有一點像 3」這類資訊。

論文同時指出，答錯的 3 大多是因為 3 這一類的偏差值學得太低：訓練時從來沒有一張圖的答案是 3。
把 3 的 logit 加上一個常數之後，大部分的 3 就答對了。論文是用測試集挑這個常數，下面照做，所以這個數字偏樂觀。

In [ ]:
mask = y_train != 3
print('移轉集張數', int(mask.sum()), '，其中數字 3：', int((y_train[mask] == 3).sum()), '張')


def threes_wrong(logits):
    return int(((logits.argmax(1) != y_test) & (y_test == 3)).sum())


for name, t, a in [('只學正確答案', 1.0, 0.0), ('蒸餾 T = 8', 8.0, 0.9)]:
    s = train_student(100, t, a, mask=mask)
    err, logits = test_errors(s)
    best = None
    for b in np.arange(0, 15.01, 0.25):
        shift = np.zeros(10, 'float32')
        shift[3] = b
        e = int(((logits + shift).argmax(1) != y_test).sum())
        if best is None or e < best[0]:
            best = (e, b, threes_wrong(logits + shift))
    results[('no3', name)] = (err, threes_wrong(logits), best)
    print(f'{name}：答錯 {err} 張，其中 3 錯 {threes_wrong(logits)} / 1010')
    print(f'    3 的 logit 加 {best[1]:.2f} 後：答錯 {best[0]} 張，其中 3 錯 {best[2]} / 1010'
          f'（3 答對 {100 * (1 - best[2] / 1010):.1f}%）')

## 8. 結果整理

本筆記本的資料與模型都比論文小，答錯張數不會和論文相同；要看的是三個趨勢是否一致。

In [ ]:
print(f'{"模型":<22}{"測試集答錯張數":>10}')
print(f'{"老師 1200-1200":<22}{teacher_err:>10}')
for units in [100, 30]:
    print(f'{"學生 %d：只學正確答案" % units:<22}{results[(units, "hard")]:>10}')
    best_t = min(TEMPS, key=lambda t: results[(units, t)])
    print(f'{"學生 %d：蒸餾最佳 T=%d" % (units, best_t):<22}{results[(units, best_t)]:>10}')

fig, ax = plt.subplots(figsize=(7, 3.2))
for units, color in [(100, '#1F4E79'), (30, '#C03030')]:
    ax.plot(TEMPS, [results[(units, t)] for t in TEMPS], 'o-', color=color,
            label=f'student {units}-{units}, distilled')
    ax.axhline(results[(units, 'hard')], color=color, ls='--', label=f'student {units}-{units}, hard labels only')
ax.set_xscale('log')
ax.set_xticks(TEMPS)
ax.set_xticklabels([str(t) for t in TEMPS])
ax.set_xlabel('temperature T')
ax.set_ylabel('test errors (of 10,000)')
ax.legend(fontsize=8)
plt.show()

| 論文的結論 | 對照本筆記本的結果 |
|---|---|
| 蒸餾的學生答錯比只學正確答案的學生少 | 第 5 段 |
| 學生很小時，中間的溫度最好 | 第 6 段 |
| 學生沒看過 3，調整 3 的偏差值後仍能認出大部分的 3 | 第 7 段 |

**練習**

1. 第 3 段把學生的 logit 改成 $(8, 4, 0)$，也就是和老師相同。軟目標梯度會是多少？為什麼？
2. 第 5 段把 $\alpha$ 改成 0.5，蒸餾的效果有什麼變化？
3. 第 7 段改成拿掉數字 7，再做一次。

---

依據：G. Hinton, O. Vinyals, J. Dean, "Distilling the Knowledge in a Neural Network," arXiv:1503.02531, 2015。
資料：MNIST 手寫數字（Y. LeCun、C. Cortes、C. J. C. Burges），本課程取訓練集 10,000 張與完整測試集。